In [ ]:
import pandas as pd
import numpy as np

# Load dataset from GitHub
url = "https://raw.githubusercontent.com/NandiniBrahmbhatt/ThirdDegree/main/ml-simulator/data/renewable_sensor_dataset.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nAsset types:")
print(df["asset_type"].value_counts())

print("\nScenarios:")
print(df["scenario"].value_counts())

print("\nMissing values:")
print(df.isna().sum())

Dataset loaded successfully!
Shape: (60480, 13)

Columns:
['asset_id', 'asset_type', 'timestamp', 'wind_speed', 'rotor_speed', 'vibration', 'temperature', 'current', 'power_output', 'solar_irradiance', 'voltage', 'soiling_level', 'scenario']

Asset types:
asset_type
solar    30240
wind     30240
Name: count, dtype: int64

Scenarios:
scenario
healthy                               34560
excessive_soiling                      4320
inverter_overheating                   4320
electrical_performance_degradation     4320
bearing_degradation                    4320
gearbox_abnormality                    4320
generator_electrical_abnormality       4320
Name: count, dtype: int64

Missing values:
asset_id                0
asset_type              0
timestamp               0
wind_speed          30240
rotor_speed         30240
vibration           30240
temperature             0
current                 0
power_output            0
solar_irradiance    30240
voltage             30240
soiling_level      

In [2]:
# =========================================================
# STEP 2: PREPARE SOLAR AND WIND DATA
# =========================================================

# Separate asset types
wind_df = df[df["asset_type"] == "wind"].copy()
solar_df = df[df["asset_type"] == "solar"].copy()

print("Wind dataset shape:", wind_df.shape)
print("Solar dataset shape:", solar_df.shape)


# ---------------------------------------------------------
# WIND FEATURES
# ---------------------------------------------------------

wind_features = [
    "wind_speed",
    "rotor_speed",
    "vibration",
    "temperature",
    "current",
    "power_output",
]


# ---------------------------------------------------------
# SOLAR FEATURES
# ---------------------------------------------------------

solar_features = [
    "solar_irradiance",
    "temperature",
    "voltage",
    "current",
    "power_output",
    "soiling_level",
]


# ---------------------------------------------------------
# HEALTHY TRAINING DATA
# ---------------------------------------------------------

wind_healthy = wind_df[
    wind_df["scenario"] == "healthy"
].copy()

solar_healthy = solar_df[
    solar_df["scenario"] == "healthy"
].copy()


print("\nHealthy wind samples:", len(wind_healthy))
print("Healthy solar samples:", len(solar_healthy))


# ---------------------------------------------------------
# MODEL INPUT MATRICES
# ---------------------------------------------------------

X_wind_train = wind_healthy[wind_features].copy()

X_solar_train = solar_healthy[solar_features].copy()


print("\nWind training matrix:", X_wind_train.shape)
print("Solar training matrix:", X_solar_train.shape)


# ---------------------------------------------------------
# CHECK FOR MISSING VALUES
# ---------------------------------------------------------

print("\nMissing values in wind training data:")
print(X_wind_train.isna().sum())

print("\nMissing values in solar training data:")
print(X_solar_train.isna().sum())

Wind dataset shape: (30240, 13)
Solar dataset shape: (30240, 13)

Healthy wind samples: 17280
Healthy solar samples: 17280

Wind training matrix: (17280, 6)
Solar training matrix: (17280, 6)

Missing values in wind training data:
wind_speed      0
rotor_speed     0
vibration       0
temperature     0
current         0
power_output    0
dtype: int64

Missing values in solar training data:
solar_irradiance    0
temperature         0
voltage             0
current             0
power_output        0
soiling_level       0
dtype: int64


In [3]:
from sklearn.ensemble import IsolationForest


# =========================================================
# STEP 3: TRAIN ISOLATION FOREST MODELS
# =========================================================

# ---------------------------------------------------------
# WIND MODEL
# ---------------------------------------------------------

wind_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

wind_model.fit(X_wind_train)


# ---------------------------------------------------------
# SOLAR MODEL
# ---------------------------------------------------------

solar_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

solar_model.fit(X_solar_train)


print("Isolation Forest models trained successfully!")

print("\nWind model:")
print(wind_model)

print("\nSolar model:")
print(solar_model)

Isolation Forest models trained successfully!

Wind model:
IsolationForest(contamination=0.05, n_estimators=200, n_jobs=-1,
                random_state=42)

Solar model:
IsolationForest(contamination=0.05, n_estimators=200, n_jobs=-1,
                random_state=42)


In [4]:
# =========================================================
# STEP 4: TEST ANOMALY DETECTION
# =========================================================

# ---------------------------------------------------------
# WIND TEST DATA
# ---------------------------------------------------------

wind_test = wind_df.copy()

X_wind_test = wind_test[wind_features]

wind_test["prediction"] = wind_model.predict(X_wind_test)

# Isolation Forest:
#   1  = normal
#  -1  = anomaly

wind_test["anomaly"] = (
    wind_test["prediction"] == -1
).astype(int)


# ---------------------------------------------------------
# SOLAR TEST DATA
# ---------------------------------------------------------

solar_test = solar_df.copy()

X_solar_test = solar_test[solar_features]

solar_test["prediction"] = solar_model.predict(X_solar_test)

solar_test["anomaly"] = (
    solar_test["prediction"] == -1
).astype(int)


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

print("WIND ANOMALY DETECTION")
print("=" * 40)

print(
    wind_test.groupby("scenario")["anomaly"]
    .agg(["count", "sum", "mean"])
)


print("\nSOLAR ANOMALY DETECTION")
print("=" * 40)

print(
    solar_test.groupby("scenario")["anomaly"]
    .agg(["count", "sum", "mean"])
)

WIND ANOMALY DETECTION
                                  count   sum      mean
scenario                                               
bearing_degradation                4320  3647  0.844213
gearbox_abnormality                4320  3343  0.773843
generator_electrical_abnormality   4320  1592  0.368519
healthy                           17280   864  0.050000

SOLAR ANOMALY DETECTION
                                    count   sum      mean
scenario                                                 
electrical_performance_degradation   4320  2944  0.681481
excessive_soiling                    4320   310  0.071759
healthy                             17280   864  0.050000
inverter_overheating                 4320   589  0.136343


In [5]:
# =========================================================
# STEP 4B: CHECK PROGRESSIVE ANOMALY DETECTION
# =========================================================

def progression_anomaly_rates(data):
    data = data.copy()

    data["progression_stage"] = pd.qcut(
        data.groupby("asset_id").cumcount(),
        q=4,
        labels=[
            "early",
            "developing",
            "advanced",
            "high_risk"
        ]
    )

    return (
        data.groupby(
            ["scenario", "progression_stage"],
            observed=True
        )["anomaly"]
        .mean()
        .mul(100)
        .round(2)
    )


print("WIND PROGRESSION")
print("=" * 50)
print(progression_anomaly_rates(wind_test))

print("\nSOLAR PROGRESSION")
print("=" * 50)
print(progression_anomaly_rates(solar_test))

WIND PROGRESSION
scenario                          progression_stage
bearing_degradation               early                 40.09
                                  developing            97.59
                                  advanced             100.00
                                  high_risk            100.00
gearbox_abnormality               early                 18.70
                                  developing            90.83
                                  advanced             100.00
                                  high_risk            100.00
generator_electrical_abnormality  early                 10.65
                                  developing             0.09
                                  advanced              37.04
                                  high_risk             99.63
healthy                           early                  8.87
                                  developing             0.00
                                  advanced               0.00
 

In [9]:
# =========================================================
# STEP 5: INSPECT ISOLATION FOREST ANOMALY SCORES
# =========================================================

# Lower decision_function values = more anomalous

wind_test["anomaly_score"] = wind_model.decision_function(
    X_wind_test
)

solar_test["anomaly_score"] = solar_model.decision_function(
    X_solar_test
)


# ---------------------------------------------------------
# WIND
# ---------------------------------------------------------

print("WIND ANOMALY SCORES")
print("=" * 60)

print(
    wind_test.groupby("scenario")["anomaly_score"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)


# ---------------------------------------------------------
# SOLAR
# ---------------------------------------------------------

print("\nSOLAR ANOMALY SCORES")
print("=" * 60)

print(
    solar_test.groupby("scenario")["anomaly_score"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

WIND ANOMALY SCORES
                                    mean     std     min     max
scenario                                                        
bearing_degradation              -0.0280  0.0270 -0.1098  0.0910
gearbox_abnormality              -0.0258  0.0363 -0.1145  0.1106
generator_electrical_abnormality  0.0201  0.0523 -0.1167  0.1193
healthy                           0.0802  0.0376 -0.1236  0.1335

SOLAR ANOMALY SCORES
                                      mean     std     min     max
scenario                                                          
electrical_performance_degradation -0.0094  0.0341 -0.0872  0.1037
excessive_soiling                   0.0590  0.0342 -0.1011  0.1064
healthy                             0.0774  0.0370 -0.1323  0.1285
inverter_overheating                0.0272  0.0278 -0.1055  0.1010


In [10]:
# =========================================================
# STEP 6: MODEL EVALUATION
# =========================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)


# ---------------------------------------------------------
# CREATE GROUND TRUTH LABELS
# ---------------------------------------------------------

wind_test["actual"] = (
    wind_test["scenario"] != "healthy"
).astype(int)

solar_test["actual"] = (
    solar_test["scenario"] != "healthy"
).astype(int)


# Model anomaly:
# 1 = anomaly
# 0 = normal

wind_test["predicted"] = wind_test["anomaly"]
solar_test["predicted"] = solar_test["anomaly"]


# ---------------------------------------------------------
# WIND EVALUATION
# ---------------------------------------------------------

print("=" * 60)
print("WIND MODEL EVALUATION")
print("=" * 60)

print("\nAccuracy:")
print(
    round(
        accuracy_score(
            wind_test["actual"],
            wind_test["predicted"]
        ),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        wind_test["actual"],
        wind_test["predicted"],
        target_names=["Healthy", "Anomaly"]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        wind_test["actual"],
        wind_test["predicted"]
    )
)


# ---------------------------------------------------------
# SOLAR EVALUATION
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("SOLAR MODEL EVALUATION")
print("=" * 60)

print("\nAccuracy:")
print(
    round(
        accuracy_score(
            solar_test["actual"],
            solar_test["predicted"]
        ),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        solar_test["actual"],
        solar_test["predicted"],
        target_names=["Healthy", "Anomaly"]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        solar_test["actual"],
        solar_test["predicted"]
    )
)

WIND MODEL EVALUATION

Accuracy:
0.8267

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.79      0.95      0.86     17280
     Anomaly       0.91      0.66      0.77     12960

    accuracy                           0.83     30240
   macro avg       0.85      0.81      0.81     30240
weighted avg       0.84      0.83      0.82     30240

Confusion Matrix:
[[16416   864]
 [ 4378  8582]]

SOLAR MODEL EVALUATION

Accuracy:
0.6699

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.64      0.95      0.77     17280
     Anomaly       0.82      0.30      0.44     12960

    accuracy                           0.67     30240
   macro avg       0.73      0.62      0.60     30240
weighted avg       0.72      0.67      0.62     30240

Confusion Matrix:
[[16416   864]
 [ 9117  3843]]


In [11]:
# =========================================================
# STEP 7: FEATURE ENGINEERING
# =========================================================

# -----------------------------
# WIND DERIVED FEATURES
# -----------------------------

wind_df["power_per_wind_speed"] = (
    wind_df["power_output"] /
    (wind_df["wind_speed"] + 1e-6)
)

wind_df["power_per_rotor_speed"] = (
    wind_df["power_output"] /
    (wind_df["rotor_speed"] + 1e-6)
)


# -----------------------------
# SOLAR DERIVED FEATURES
# -----------------------------

solar_df["power_per_irradiance"] = (
    solar_df["power_output"] /
    (solar_df["solar_irradiance"] + 1e-6)
)

solar_df["current_per_irradiance"] = (
    solar_df["current"] /
    (solar_df["solar_irradiance"] + 1e-6)
)


# -----------------------------
# UPDATED FEATURE LISTS
# -----------------------------

wind_features_engineered = [
    "wind_speed",
    "rotor_speed",
    "vibration",
    "temperature",
    "current",
    "power_output",
    "power_per_wind_speed",
    "power_per_rotor_speed",
]

solar_features_engineered = [
    "solar_irradiance",
    "temperature",
    "voltage",
    "current",
    "power_output",
    "soiling_level",
    "power_per_irradiance",
    "current_per_irradiance",
]


# -----------------------------
# HEALTHY TRAINING DATA
# -----------------------------

wind_healthy_engineered = wind_df[
    wind_df["scenario"] == "healthy"
]

solar_healthy_engineered = solar_df[
    solar_df["scenario"] == "healthy"
]


X_wind_train_engineered = (
    wind_healthy_engineered[
        wind_features_engineered
    ]
)

X_solar_train_engineered = (
    solar_healthy_engineered[
        solar_features_engineered
    ]
)


print("Feature engineering complete!")

print("\nWind features:")
print(wind_features_engineered)

print("\nSolar features:")
print(solar_features_engineered)

print(
    "\nWind training shape:",
    X_wind_train_engineered.shape
)

print(
    "Solar training shape:",
    X_solar_train_engineered.shape
)

Feature engineering complete!

Wind features:
['wind_speed', 'rotor_speed', 'vibration', 'temperature', 'current', 'power_output', 'power_per_wind_speed', 'power_per_rotor_speed']

Solar features:
['solar_irradiance', 'temperature', 'voltage', 'current', 'power_output', 'soiling_level', 'power_per_irradiance', 'current_per_irradiance']

Wind training shape: (17280, 8)
Solar training shape: (17280, 8)


In [12]:
# =========================================================
# STEP 8: TRAIN ENGINEERED ISOLATION FOREST MODELS
# =========================================================

wind_model_engineered = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

solar_model_engineered = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)


# Train only on healthy data
wind_model_engineered.fit(X_wind_train_engineered)
solar_model_engineered.fit(X_solar_train_engineered)


print("Engineered Isolation Forest models trained successfully!")

Engineered Isolation Forest models trained successfully!


In [16]:
# =========================================================
# STEP 9: EVALUATE ENGINEERED MODELS
# =========================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)


# ---------------------------------------------------------
# PREDICTIONS
# ---------------------------------------------------------

wind_test["engineered_prediction"] = (
    wind_model_engineered.predict(
        wind_test[wind_features_engineered]
    )
)

solar_test["engineered_prediction"] = (
    solar_model_engineered.predict(
        solar_test[solar_features_engineered]
    )
)


# Convert Isolation Forest output:
#  1  = normal
# -1  = anomaly

wind_test["engineered_anomaly"] = (
    wind_test["engineered_prediction"] == -1
).astype(int)

solar_test["engineered_anomaly"] = (
    solar_test["engineered_prediction"] == -1
).astype(int)


# ---------------------------------------------------------
# GROUND TRUTH
# ---------------------------------------------------------

wind_test["actual"] = (
    wind_test["scenario"] != "healthy"
).astype(int)

solar_test["actual"] = (
    solar_test["scenario"] != "healthy"
).astype(int)


# ---------------------------------------------------------
# WIND EVALUATION
# ---------------------------------------------------------

print("=" * 60)
print("ENGINEERED WIND MODEL")
print("=" * 60)

print(
    classification_report(
        wind_test["actual"],
        wind_test["engineered_anomaly"],
        target_names=["Healthy", "Anomaly"]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        wind_test["actual"],
        wind_test["engineered_anomaly"]
    )
)


# ---------------------------------------------------------
# SOLAR EVALUATION
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("ENGINEERED SOLAR MODEL")
print("=" * 60)

print(
    classification_report(
        solar_test["actual"],
        solar_test["engineered_anomaly"],
        target_names=["Healthy", "Anomaly"]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        solar_test["actual"],
        solar_test["engineered_anomaly"]
    )
)


# ---------------------------------------------------------
# ACCURACY SUMMARY
# ---------------------------------------------------------

wind_accuracy = accuracy_score(
    wind_test["actual"],
    wind_test["engineered_anomaly"]
)

solar_accuracy = accuracy_score(
    solar_test["actual"],
    solar_test["engineered_anomaly"]
)

print("\n" + "=" * 60)
print("ACCURACY SUMMARY")
print("=" * 60)

print(f"Wind accuracy:  {wind_accuracy:.4f}")
print(f"Solar accuracy: {solar_accuracy:.4f}")

ENGINEERED WIND MODEL
              precision    recall  f1-score   support

     Healthy       0.80      0.95      0.87     17280
     Anomaly       0.91      0.68      0.78     12960

    accuracy                           0.83     30240
   macro avg       0.85      0.81      0.82     30240
weighted avg       0.85      0.83      0.83     30240

Confusion Matrix:
[[16416   864]
 [ 4148  8812]]

ENGINEERED SOLAR MODEL
              precision    recall  f1-score   support

     Healthy       0.67      0.95      0.79     17280
     Anomaly       0.85      0.38      0.53     12960

    accuracy                           0.71     30240
   macro avg       0.76      0.67      0.66     30240
weighted avg       0.75      0.71      0.68     30240

Confusion Matrix:
[[16416   864]
 [ 8026  4934]]

ACCURACY SUMMARY
Wind accuracy:  0.8343
Solar accuracy: 0.7060


In [15]:
# =========================================================
# PREPARE TEST DATA WITH ENGINEERED FEATURES
# =========================================================

wind_test = wind_df.copy()
solar_test = solar_df.copy()

print("Wind test features ready:", wind_test[wind_features_engineered].shape)
print("Solar test features ready:", solar_test[solar_features_engineered].shape)

Wind test features ready: (30240, 8)
Solar test features ready: (30240, 8)


In [24]:
# =========================================================
# FINAL TEST DATA + ENGINEERED FEATURES + ANOMALY SCORES
# =========================================================

# Define FINAL feature lists explicitly
wind_features_engineered = [
    "wind_speed",
    "rotor_speed",
    "vibration",
    "temperature",
    "current",
    "power_output",
    "power_per_wind_speed",
    "power_per_rotor_speed"
]

solar_features_engineered = [
    "solar_irradiance",
    "temperature",
    "voltage",
    "current",
    "power_output",
    "soiling_level",
    "power_per_irradiance",
    "current_per_irradiance"
]

# Fresh test data
wind_test = wind_df.copy()
solar_test = solar_df.copy()

# Wind engineered features
wind_test["power_per_wind_speed"] = (
    wind_test["power_output"] /
    wind_test["wind_speed"].replace(0, np.nan)
)

wind_test["power_per_rotor_speed"] = (
    wind_test["power_output"] /
    wind_test["rotor_speed"].replace(0, np.nan)
)

# Solar engineered features
solar_test["power_per_irradiance"] = (
    solar_test["power_output"] /
    solar_test["solar_irradiance"].replace(0, np.nan)
)

solar_test["current_per_irradiance"] = (
    solar_test["current"] /
    solar_test["solar_irradiance"].replace(0, np.nan)
)

# Clean invalid values
wind_test[wind_features_engineered] = (
    wind_test[wind_features_engineered]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

solar_test[solar_features_engineered] = (
    solar_test[solar_features_engineered]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# FINAL engineered-model anomaly scores
wind_test["anomaly_score"] = wind_model_engineered.decision_function(
    wind_test[wind_features_engineered]
)

solar_test["anomaly_score"] = solar_model_engineered.decision_function(
    solar_test[solar_features_engineered]
)

print("FINAL engineered anomaly scores restored.")
print("Wind features:", len(wind_features_engineered))
print("Solar features:", len(solar_features_engineered))
print("Wind test shape:", wind_test[wind_features_engineered].shape)
print("Solar test shape:", solar_test[solar_features_engineered].shape)

FINAL engineered anomaly scores restored.
Wind features: 8
Solar features: 8
Wind test shape: (30240, 8)
Solar test shape: (30240, 8)


In [25]:
# =========================================================
# STEP 10: HEALTH AND RISK SCORING
# =========================================================

def calculate_risk_score(anomaly_score):
    """
    Convert Isolation Forest anomaly score into
    a deterministic 0-100 risk score.

    Lower anomaly score = higher risk.
    """

    # Expected useful score range for our models
    min_score = -0.12
    max_score = 0.13

    normalized = (
        (max_score - anomaly_score)
        / (max_score - min_score)
    )

    normalized = np.clip(
        normalized,
        0,
        1
    )

    return normalized * 100


def calculate_health_score(risk_score):
    """Health is the inverse of risk."""

    return 100 - risk_score


def get_risk_status(risk_score):

    if risk_score < 25:
        return "Healthy"

    elif risk_score < 50:
        return "Low Risk"

    elif risk_score < 75:
        return "Medium Risk"

    else:
        return "High Risk"


# ---------------------------------------------------------
# APPLY TO WIND
# ---------------------------------------------------------

wind_test["risk_score"] = wind_test[
    "anomaly_score"
].apply(calculate_risk_score)

wind_test["health_score"] = wind_test[
    "risk_score"
].apply(calculate_health_score)

wind_test["status"] = wind_test[
    "risk_score"
].apply(get_risk_status)


# ---------------------------------------------------------
# APPLY TO SOLAR
# ---------------------------------------------------------

solar_test["risk_score"] = solar_test[
    "anomaly_score"
].apply(calculate_risk_score)

solar_test["health_score"] = solar_test[
    "risk_score"
].apply(calculate_health_score)

solar_test["status"] = solar_test[
    "risk_score"
].apply(get_risk_status)


# ---------------------------------------------------------
# DISPLAY EXAMPLES
# ---------------------------------------------------------

print("WIND HEALTH/RISK EXAMPLES")
print("=" * 60)

print(
    wind_test[
        [
            "asset_id",
            "timestamp",
            "scenario",
            "anomaly_score",
            "risk_score",
            "health_score",
            "status",
        ]
    ].head(10).to_string(index=False)
)


print("\nSOLAR HEALTH/RISK EXAMPLES")
print("=" * 60)

print(
    solar_test[
        [
            "asset_id",
            "timestamp",
            "scenario",
            "anomaly_score",
            "risk_score",
            "health_score",
            "status",
        ]
    ].head(10).to_string(index=False)
)

WIND HEALTH/RISK EXAMPLES
asset_id           timestamp scenario  anomaly_score  risk_score  health_score      status
  WT-001 2026-01-01 00:00:00  healthy       0.005631   49.747728     50.252272    Low Risk
  WT-001 2026-01-01 00:10:00  healthy      -0.035947   66.378651     33.621349 Medium Risk
  WT-001 2026-01-01 00:20:00  healthy      -0.004673   53.869267     46.130733 Medium Risk
  WT-001 2026-01-01 00:30:00  healthy      -0.004522   53.808666     46.191334 Medium Risk
  WT-001 2026-01-01 00:40:00  healthy      -0.102486   92.994274      7.005726   High Risk
  WT-001 2026-01-01 00:50:00  healthy      -0.061471   76.588321     23.411679   High Risk
  WT-001 2026-01-01 01:00:00  healthy      -0.024758   61.903135     38.096865 Medium Risk
  WT-001 2026-01-01 01:10:00  healthy       0.023841   42.463771     57.536229    Low Risk
  WT-001 2026-01-01 01:20:00  healthy       0.019569   44.172292     55.827708    Low Risk
  WT-001 2026-01-01 01:30:00  healthy      -0.070587   80.234690

In [26]:
# =========================================================
# STEP 11 — PROBABLE ISSUE / PATTERN RULES
# =========================================================

def diagnose_wind(row):
    """
    Rule-based interpretation of an anomalous wind reading.
    Returns a stable issue code, probable issue,
    contributing factors, and recommendation.
    """

    factors = []

    # Bearing degradation
    if (
        row["vibration"] > 1.5
        and row["temperature"] > 65
    ):
        factors = [
            "Elevated vibration",
            "Increased temperature"
        ]
        return {
            "issue_code": "WIND_BEARING",
            "probable_issue": "Possible bearing degradation",
            "contributing_factors": factors,
            "recommendation": "Inspect bearing condition and schedule maintenance."
        }

    # Gearbox abnormality
    if (
        row["vibration"] > 1.8
        and row["rotor_speed"] > 20
    ):
        factors = [
            "High vibration",
            "Elevated rotor speed"
        ]
        return {
            "issue_code": "WIND_GEARBOX",
            "probable_issue": "Possible gearbox abnormality",
            "contributing_factors": factors,
            "recommendation": "Inspect gearbox condition and check drivetrain components."
        }

    # Generator/electrical abnormality
    if (
        row["current"] > 80
        and row["power_output"] < 5000
    ):
        factors = [
            "High electrical current",
            "Lower-than-expected power output"
        ]
        return {
            "issue_code": "WIND_GENERATOR",
            "probable_issue": "Possible generator/electrical abnormality",
            "contributing_factors": factors,
            "recommendation": "Inspect generator electrical performance and connections."
        }

    # Generic anomaly
    return {
        "issue_code": "WIND_GENERAL",
        "probable_issue": "Possible abnormal operating condition",
        "contributing_factors": ["Sensor behaviour differs from learned healthy patterns"],
        "recommendation": "Inspect recent sensor trends and schedule further diagnostics."
    }


def diagnose_solar(row):
    """
    Rule-based interpretation of an anomalous solar reading.
    Returns a stable issue code, probable issue,
    contributing factors, and recommendation.
    """

    # Excessive soiling
    if (
        row["soiling_level"] > 0.5
        and row["power_per_irradiance"] < 10
    ):
        return {
            "issue_code": "SOLAR_SOILING",
            "probable_issue": "Possible excessive panel soiling",
            "contributing_factors": [
                "High soiling level",
                "Reduced power relative to irradiance"
            ],
            "recommendation": "Inspect and clean the solar panels."
        }

    # Inverter overheating
    if row["temperature"] > 75:
        return {
            "issue_code": "SOLAR_OVERHEATING",
            "probable_issue": "Possible inverter overheating",
            "contributing_factors": [
                "Elevated operating temperature"
            ],
            "recommendation": "Inspect inverter cooling and thermal conditions."
        }

    # Electrical performance degradation
    if (
        row["voltage"] < 370
        and row["power_per_irradiance"] < 12
    ):
        return {
            "issue_code": "SOLAR_ELECTRICAL",
            "probable_issue": "Possible electrical performance degradation",
            "contributing_factors": [
                "Reduced voltage",
                "Reduced power relative to irradiance"
            ],
            "recommendation": "Inspect inverter electrical performance and connections."
        }

    # Generic anomaly
    return {
        "issue_code": "SOLAR_GENERAL",
        "probable_issue": "Possible abnormal operating condition",
        "contributing_factors": ["Sensor behaviour differs from learned healthy patterns"],
        "recommendation": "Inspect recent sensor trends and schedule further diagnostics."
    }


print("Pattern diagnosis rules created successfully.")

Pattern diagnosis rules created successfully.


In [27]:
# =========================================================
# STEP 12 — FINAL PREDICTION PIPELINE
# =========================================================

def predict_asset(row):
    """
    Run the complete predictive-maintenance pipeline
    for a single sensor reading.
    """

    asset_type = row["asset_type"]

    # -----------------------------------------------------
    # WIND
    # -----------------------------------------------------
    if asset_type == "wind":

        power_per_wind_speed = (
            row["power_output"] / row["wind_speed"]
            if row["wind_speed"] != 0 else 0
        )

        power_per_rotor_speed = (
            row["power_output"] / row["rotor_speed"]
            if row["rotor_speed"] != 0 else 0
        )

        features = pd.DataFrame([{
            "wind_speed": row["wind_speed"],
            "rotor_speed": row["rotor_speed"],
            "vibration": row["vibration"],
            "temperature": row["temperature"],
            "current": row["current"],
            "power_output": row["power_output"],
            "power_per_wind_speed": power_per_wind_speed,
            "power_per_rotor_speed": power_per_rotor_speed
        }])

        anomaly_score = wind_model_engineered.decision_function(
            features[wind_features_engineered]
        )[0]

        anomaly = int(
            wind_model_engineered.predict(
                features[wind_features_engineered]
            )[0] == -1
        )

        diagnosis = diagnose_wind(row)

    # -----------------------------------------------------
    # SOLAR
    # -----------------------------------------------------
    elif asset_type == "solar":

        power_per_irradiance = (
            row["power_output"] / row["solar_irradiance"]
            if row["solar_irradiance"] != 0 else 0
        )

        current_per_irradiance = (
            row["current"] / row["solar_irradiance"]
            if row["solar_irradiance"] != 0 else 0
        )

        features = pd.DataFrame([{
            "solar_irradiance": row["solar_irradiance"],
            "temperature": row["temperature"],
            "voltage": row["voltage"],
            "current": row["current"],
            "power_output": row["power_output"],
            "soiling_level": row["soiling_level"],
            "power_per_irradiance": power_per_irradiance,
            "current_per_irradiance": current_per_irradiance
        }])

        anomaly_score = solar_model_engineered.decision_function(
            features[solar_features_engineered]
        )[0]

        anomaly = int(
            solar_model_engineered.predict(
                features[solar_features_engineered]
            )[0] == -1
        )

        diagnosis = diagnose_solar(row)

    else:
        raise ValueError("asset_type must be 'wind' or 'solar'")

    # -----------------------------------------------------
    # HEALTH / RISK
    # -----------------------------------------------------

    min_score = -0.12
    max_score = 0.13

    normalized_risk = (
        (max_score - anomaly_score)
        / (max_score - min_score)
    )

    risk_score = float(
        np.clip(normalized_risk, 0, 1) * 100
    )

    health_score = 100 - risk_score

    if risk_score < 25:
        status = "Healthy"
    elif risk_score < 50:
        status = "Low Risk"
    elif risk_score < 75:
        status = "Medium Risk"
    else:
        status = "High Risk"

    # -----------------------------------------------------
    # FINAL OUTPUT
    # -----------------------------------------------------

    return {
        "asset_id": row["asset_id"],
        "asset_type": asset_type,
        "timestamp": row["timestamp"],
        "anomaly": anomaly,
        "anomaly_score": round(float(anomaly_score), 4),
        "risk_score": round(risk_score, 2),
        "health_score": round(health_score, 2),
        "status": status,
        "issue_code": diagnosis["issue_code"],
        "probable_issue": diagnosis["probable_issue"],
        "contributing_factors": diagnosis["contributing_factors"],
        "recommendation": diagnosis["recommendation"]
    }


print("Final prediction pipeline created successfully.")

Final prediction pipeline created successfully.


In [28]:
# =========================================================
# STEP 13 — EXPORT FINAL MODELS
# =========================================================

import joblib
import os

# Create models directory
os.makedirs("models", exist_ok=True)

# Export final engineered models
joblib.dump(
    wind_model_engineered,
    "models/wind_isolation_forest.joblib"
)

joblib.dump(
    solar_model_engineered,
    "models/solar_isolation_forest.joblib"
)

print("Models exported successfully!")
print("Wind model: models/wind_isolation_forest.joblib")
print("Solar model: models/solar_isolation_forest.joblib")

Models exported successfully!
Wind model: models/wind_isolation_forest.joblib
Solar model: models/solar_isolation_forest.joblib


In [29]:
# =========================================================
# STEP 14 — VERIFY EXPORTED MODELS
# =========================================================

import joblib

# Load exported models
wind_model_loaded = joblib.load(
    "models/wind_isolation_forest.joblib"
)

solar_model_loaded = joblib.load(
    "models/solar_isolation_forest.joblib"
)

# Test that the loaded models can make predictions
wind_prediction = wind_model_loaded.predict(
    wind_test[wind_features_engineered].head(5)
)

solar_prediction = solar_model_loaded.predict(
    solar_test[solar_features_engineered].head(5)
)

print("Export verification successful!")
print("Wind predictions:", wind_prediction)
print("Solar predictions:", solar_prediction)

Export verification successful!
Wind predictions: [ 1 -1 -1 -1 -1]
Solar predictions: [-1  1 -1 -1 -1]


In [30]:
# =========================================================
# STEP 15 — MODEL INPUT / OUTPUT CONTRACT
# =========================================================

model_contract = {
    "models": {
        "wind": {
            "model_file": "wind_isolation_forest.joblib",
            "features": wind_features_engineered
        },
        "solar": {
            "model_file": "solar_isolation_forest.joblib",
            "features": solar_features_engineered
        }
    },

    "input": {
        "asset_id": "string",
        "asset_type": "wind | solar",
        "timestamp": "ISO-8601 timestamp",

        "wind_sensors": {
            "wind_speed": "float",
            "rotor_speed": "float",
            "vibration": "float",
            "temperature": "float",
            "current": "float",
            "power_output": "float"
        },

        "solar_sensors": {
            "solar_irradiance": "float",
            "temperature": "float",
            "voltage": "float",
            "current": "float",
            "power_output": "float",
            "soiling_level": "float"
        }
    },

    "output": {
        "asset_id": "string",
        "asset_type": "wind | solar",
        "timestamp": "ISO-8601 timestamp",
        "anomaly": "0 = normal, 1 = anomaly",
        "anomaly_score": "float",
        "risk_score": "0-100 indicator",
        "health_score": "0-100 indicator",
        "status": "Healthy | Low Risk | Medium Risk | High Risk",
        "issue_code": "string",
        "probable_issue": "string",
        "contributing_factors": "list of strings",
        "recommendation": "string"
    }
}

print("MODEL INPUT / OUTPUT CONTRACT")
print("=" * 60)

print("\nWind features:")
for feature in wind_features_engineered:
    print("-", feature)

print("\nSolar features:")
for feature in solar_features_engineered:
    print("-", feature)

print("\nOutput fields:")
for field in model_contract["output"]:
    print("-", field)

print("\nContract created successfully.")

MODEL INPUT / OUTPUT CONTRACT

Wind features:
- wind_speed
- rotor_speed
- vibration
- temperature
- current
- power_output
- power_per_wind_speed
- power_per_rotor_speed

Solar features:
- solar_irradiance
- temperature
- voltage
- current
- power_output
- soiling_level
- power_per_irradiance
- current_per_irradiance

Output fields:
- asset_id
- asset_type
- timestamp
- anomaly
- anomaly_score
- risk_score
- health_score
- status
- issue_code
- probable_issue
- contributing_factors
- recommendation

Contract created successfully.


In [31]:
# =========================================================
# STEP 16 — FINAL END-TO-END SANITY TEST
# =========================================================

# Test one wind reading
wind_result = predict_asset(
    wind_df.iloc[0].to_dict()
)

# Test one solar reading
solar_result = predict_asset(
    solar_df.iloc[0].to_dict()
)

print("WIND PREDICTION")
print("=" * 60)
for key, value in wind_result.items():
    print(f"{key}: {value}")

print("\nSOLAR PREDICTION")
print("=" * 60)
for key, value in solar_result.items():
    print(f"{key}: {value}")

print("\nFinal end-to-end pipeline test completed successfully.")

WIND PREDICTION
asset_id: WT-001
asset_type: wind
timestamp: 2026-01-01 00:00:00
anomaly: 0
anomaly_score: 0.0056
risk_score: 49.75
health_score: 50.25
status: Low Risk
issue_code: WIND_GENERAL
probable_issue: Possible abnormal operating condition
contributing_factors: ['Sensor behaviour differs from learned healthy patterns']
recommendation: Inspect recent sensor trends and schedule further diagnostics.

SOLAR PREDICTION
asset_id: INV-001
asset_type: solar
timestamp: 2026-01-01 00:00:00
anomaly: 1
anomaly_score: -0.1254
risk_score: 100.0
health_score: 0.0
status: High Risk
issue_code: SOLAR_GENERAL
probable_issue: Possible abnormal operating condition
contributing_factors: ['Sensor behaviour differs from learned healthy patterns']
recommendation: Inspect recent sensor trends and schedule further diagnostics.

Final end-to-end pipeline test completed successfully.
